#Import

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#Create Flag Parameter

In [0]:
# Check whether the Gold dimension table already exists
if spark.catalog.tableExists("store_catalog.gold.dim_products"):
    incremental_flag = "1"
else:
    incremental_flag = "0"

In [0]:
print(incremental_flag)

1


#Create Dimension products

###Fetch Relative Columns

In [0]:
# Load distinct product records from the Gold layer
df_scr = spark.sql('''
SELECT
    --ROW_NUMBER() OVER (ORDER BY pn.start_date, pn.product_number) AS product_surrogate_key, 
    pn.product_id,
    pn.product_number,
    pn.product_name,
    pn.category_id,
    pc.category,
    pc.subcategory,
    pc.maintenance_flag,
    pn.product_line,
    pn.start_date
    --pn.crm_prd_info_surrogate_key,
    --pc.erp_PX_CAT_G1V2_surrogate_key
FROM store_catalog.gold.dim_crm_prd_info AS pn
LEFT JOIN store_catalog.gold.dim_erp_px_cat_g1v2 AS pc
    ON pn.category_id = pc.category_id
--WHERE pn.end_date IS NULL; -- Filter out all historical data
''')

In [0]:
df_scr.limit(10).display()

product_id,product_number,product_name,category_id,category,subcategory,maintenance_flag,product_line,start_date
559,CH-0234,Chain,CO_CH,Components,Chains,true,n/a,2013-07-01
315,BK-R68R-58,Road-450 Red- 58,BI_RB,Bikes,Road Bikes,true,Road,2011-07-01
374,BK-R89B-44,Road-250 Black- 44,BI_RB,Bikes,Road Bikes,true,Road,2013-07-01
602,BB-8107,ML Bottom Bracket,CO_BB,Components,Bottom Brackets,true,n/a,2013-07-01
287,FR-R38B-52,LL Road Frame - Black- 52,CO_RF,Components,Road Frames,true,Road,2013-07-01
456,TG-W091-S,Women's Tights- S,CL_TI,Clothing,Tights,false,Other Sales,2012-07-01
463,GL-H102-S,Half-Finger Gloves- S,CL_GL,Clothing,Gloves,false,Other Sales,2013-07-01
503,FR-T67U-54,LL Touring Frame - Blue- 54,CO_TF,Components,Touring Frames,true,Touring,2013-07-01
511,FR-M63S-40,ML Mountain Frame-W - Silver- 40,CO_MF,Components,Mountain Frames,true,Mountain,2013-07-01
373,BK-R89B-44,Road-250 Black- 44,BI_RB,Bikes,Road Bikes,true,Road,2012-07-01


###dim_products - Initial and Incremental

In [0]:
# Read existing Gold records or create an empty DataFrame for the initial load
if spark.catalog.tableExists("store_catalog.gold.dim_products") == True:
    df_sink = spark.sql('''
            SELECT
            product_surrogate_key,
            product_id,
            product_number,
            product_name,
            category_id,
            category,
            subcategory,
            maintenance_flag,
            product_line,
            start_date
            FROM
                store_catalog.gold.dim_products
        ''')
else:
    df_sink = spark.sql('''
        SELECT
            1 as product_surrogate_key, 
            pn.product_id,
            pn.product_number,
            pn.product_name,
            pn.category_id,
            pc.category,
            pc.subcategory,
            pc.maintenance_flag,
            pn.product_line,
            pn.start_date
        FROM store_catalog.gold.dim_crm_prd_info AS pn
        LEFT JOIN store_catalog.gold.dim_erp_px_cat_g1v2 AS pc
            ON pn.category_id = pc.category_id
        WHERE
            1 = 0
        ''')

In [0]:
df_sink.groupBy("product_id") \
    .count() \
    .filter(col("count") > 1) \
    .show(truncate=False)

+----------+-----+
|product_id|count|
+----------+-----+
+----------+-----+



###Filtering new records and old records

In [0]:
# Match Gold records with existing Gold records by product ID
df_filter = df_scr.join(df_sink,df_scr["product_id"] == df_sink["product_id"], "left")\
        .select(df_scr["product_id"],df_scr["product_number"],df_scr["product_name"],
            df_scr["category_id"],df_scr["category"],df_scr["subcategory"],
            df_scr["maintenance_flag"],df_scr["product_line"],df_scr["start_date"],
            df_sink["product_surrogate_key"]
    )

In [0]:
df_filter.groupBy("product_id") \
    .count() \
    .filter(col("count") > 1) \
    .show(truncate=False)

+----------+-----+
|product_id|count|
+----------+-----+
+----------+-----+



**df_filter_old**

In [0]:
# Separate existing product records
df_filter_old = df_filter.filter(col("product_surrogate_key").isNotNull())

**df_filter_new**

In [0]:
# Select new products that require surrogate keys
df_filter_new = df_filter.filter(col("product_surrogate_key").isNull())\
    .select(df_scr["product_id"],df_scr["product_number"],df_scr["product_name"],
            df_scr["category_id"],df_scr["category"],df_scr["subcategory"],
            df_scr["maintenance_flag"],df_scr["product_line"],df_scr["start_date"]
    )

In [0]:
df_filter_new.display()

product_id,product_number,product_name,category_id,category,subcategory,maintenance_flag,product_line,start_date


###Create Surrogate Key

**Fetch the max Surrogate Key from existing table**

In [0]:
# Retrieve the next available surrogate key
if (incremental_flag == "0"):
    max_value = 1
else:
    max_value_df = spark.sql('''SELECT MAX(product_surrogate_key) FROM store_catalog.gold.dim_products''')
    max_value = max_value_df.collect()[0][0]+1

In [0]:
print("incremental_flag:", incremental_flag)
print("max_value:", max_value)

incremental_flag: 1
max_value: 398


**Create Surrogate Key column and Add the max surrogate key**

In [0]:
# Assign unique surrogate keys to new product records
df_filter_new = df_filter_new.withColumn("product_surrogate_key",max_value + monotonically_increasing_id())
df_filter_new.display()

product_id,product_number,product_name,category_id,category,subcategory,maintenance_flag,product_line,start_date,product_surrogate_key


###Create Final DF - df_filter_old + df_filter_new

In [0]:
# Combine new products with existing product records
df_final = df_filter_new.union(df_filter_old)

In [0]:
df_final.display()

product_id,product_number,product_name,category_id,category,subcategory,maintenance_flag,product_line,start_date,product_surrogate_key
559,CH-0234,Chain,CO_CH,Components,Chains,true,n/a,2013-07-01,1
315,BK-R68R-58,Road-450 Red- 58,BI_RB,Bikes,Road Bikes,true,Road,2011-07-01,2
374,BK-R89B-44,Road-250 Black- 44,BI_RB,Bikes,Road Bikes,true,Road,2013-07-01,3
602,BB-8107,ML Bottom Bracket,CO_BB,Components,Bottom Brackets,true,n/a,2013-07-01,4
287,FR-R38B-52,LL Road Frame - Black- 52,CO_RF,Components,Road Frames,true,Road,2013-07-01,5
456,TG-W091-S,Women's Tights- S,CL_TI,Clothing,Tights,false,Other Sales,2012-07-01,6
463,GL-H102-S,Half-Finger Gloves- S,CL_GL,Clothing,Gloves,false,Other Sales,2013-07-01,7
503,FR-T67U-54,LL Touring Frame - Blue- 54,CO_TF,Components,Touring Frames,true,Touring,2013-07-01,8
511,FR-M63S-40,ML Mountain Frame-W - Silver- 40,CO_MF,Components,Mountain Frames,true,Mountain,2013-07-01,9
373,BK-R89B-44,Road-250 Black- 44,BI_RB,Bikes,Road Bikes,true,Road,2012-07-01,10


#SCD TYPE - 1 (UPSERT)

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("store_catalog.gold.dim_products"):
    # Incremental Run
    # Merge product records into the Gold dimension table
    delta_tbl = DeltaTable.forPath(spark,"abfss://gold@storenapatdatalake.dfs.core.windows.net/dim_products")

    delta_tbl.alias("trg").merge(df_final.alias("src"),"trg.product_surrogate_key = src.product_surrogate_key")\
                            .whenMatchedUpdateAll()\
                            .whenNotMatchedInsertAll()\
                            .execute()

else:
    # Initial Run
    # Create the Gold dimension table during the initial load
    df_final.write.format("delta")\
        .mode("overwrite")\
        .option("path","abfss://gold@storenapatdatalake.dfs.core.windows.net/dim_products")\
        .saveAsTable("store_catalog.gold.dim_products")

In [0]:
%sql
SELECT * FROM store_catalog.gold.dim_products ORDER BY product_surrogate_key DESC

product_id,product_number,product_name,category_id,category,subcategory,maintenance_flag,product_line,start_date,product_surrogate_key
526,FR-M21S-48,LL Mountain Frame - Silver- 48,CO_MF,Components,Mountain Frames,true,Mountain,2013-07-01,397
549,FR-M63S-38,ML Mountain Frame-W - Silver- 38,CO_MF,Components,Mountain Frames,true,Mountain,2013-07-01,396
504,FR-T67U-58,LL Touring Frame - Blue- 58,CO_TF,Components,Touring Frames,true,Touring,2013-07-01,395
370,BK-R89R-52,Road-250 Red- 52,BI_RB,Bikes,Road Bikes,true,Road,2012-07-01,394
224,CA-1098,AWC Logo Cap,CL_CA,Clothing,Caps,false,Other Sales,2012-07-01,393
464,GL-H102-M,Half-Finger Gloves- M,CL_GL,Clothing,Gloves,false,Other Sales,2012-07-01,392
275,FR-R72R-48,ML Road Frame - Red- 48,CO_RF,Components,Road Frames,true,Road,2011-07-01,391
367,BK-M47B-48,Mountain-300 Black- 48,BI_MB,Bikes,Mountain Bikes,true,Mountain,2012-07-01,390
277,FR-R72R-58,ML Road Frame - Red- 58,CO_RF,Components,Road Frames,true,Road,2011-07-01,389
257,FR-R38B-60,LL Road Frame - Black- 60,CO_RF,Components,Road Frames,true,Road,2012-07-01,388


In [0]:
%sql
SELECT * FROM store_catalog.gold.dim_products  ORDER BY product_surrogate_key DESC 

product_id,product_number,product_name,category_id,category,subcategory,maintenance_flag,product_line,start_date,product_surrogate_key
526,FR-M21S-48,LL Mountain Frame - Silver- 48,CO_MF,Components,Mountain Frames,true,Mountain,2013-07-01,397
549,FR-M63S-38,ML Mountain Frame-W - Silver- 38,CO_MF,Components,Mountain Frames,true,Mountain,2013-07-01,396
504,FR-T67U-58,LL Touring Frame - Blue- 58,CO_TF,Components,Touring Frames,true,Touring,2013-07-01,395
370,BK-R89R-52,Road-250 Red- 52,BI_RB,Bikes,Road Bikes,true,Road,2012-07-01,394
224,CA-1098,AWC Logo Cap,CL_CA,Clothing,Caps,false,Other Sales,2012-07-01,393
464,GL-H102-M,Half-Finger Gloves- M,CL_GL,Clothing,Gloves,false,Other Sales,2012-07-01,392
275,FR-R72R-48,ML Road Frame - Red- 48,CO_RF,Components,Road Frames,true,Road,2011-07-01,391
367,BK-M47B-48,Mountain-300 Black- 48,BI_MB,Bikes,Mountain Bikes,true,Mountain,2012-07-01,390
277,FR-R72R-58,ML Road Frame - Red- 58,CO_RF,Components,Road Frames,true,Road,2011-07-01,389
257,FR-R38B-60,LL Road Frame - Black- 60,CO_RF,Components,Road Frames,true,Road,2012-07-01,388


In [0]:
%sql
SELECT COUNT(*) FROM store_catalog.gold.dim_products 

count(1)
397
